# 2. Training - Mixture of Experts



In [14]:
import os, numpy as np, pandas as pd, joblib, pickle
from sklearn.base import clone
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, ExtraTreesClassifier
from sklearn.neural_network import MLPClassifier
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score, classification_report
)
from xgboost import XGBClassifier

base_dir = os.path.join('..')
splits_dir = os.path.join(base_dir, 'data', 'splits')
models_dir = os.path.join(base_dir, 'models')
SEED = 42
N_SPLITS = 5

In [15]:
train_df = pd.read_csv(os.path.join(splits_dir, 'train.csv'))
val_df = pd.read_csv(os.path.join(splits_dir, 'val.csv'))
test_df = pd.read_csv(os.path.join(splits_dir, 'test.csv'))

selected_features = joblib.load(os.path.join(models_dir, 'selected_features.pkl'))
feature_sets = joblib.load(os.path.join(models_dir, 'expert_feature_sets.pkl'))
le = joblib.load(os.path.join(models_dir, 'label_encoder.pkl'))
class_names = le.classes_.tolist()
num_classes = len(class_names)

print(class_names)
print({k: len(v) for k, v in feature_sets.items()})

['Analysis', 'Backdoor', 'DoS', 'Exploits', 'Fuzzers', 'Generic', 'Normal', 'Reconnaissance', 'Shellcode', 'Worms']
{'expert1': 15, 'expert2': 30, 'expert3': 30}


In [16]:
X_train = train_df[selected_features].values
y_train = train_df['label'].values.astype(int)

X_val = val_df[selected_features].values
y_val = val_df['label'].values.astype(int)

X_test = test_df[selected_features].values
y_test = test_df['label'].values.astype(int)

X_train_e1 = train_df[feature_sets['expert1']].values
X_train_e2 = train_df[feature_sets['expert2']].values
X_train_e3 = train_df[feature_sets['expert3']].values

X_val_e1 = val_df[feature_sets['expert1']].values
X_val_e2 = val_df[feature_sets['expert2']].values
X_val_e3 = val_df[feature_sets['expert3']].values

X_test_e1 = test_df[feature_sets['expert1']].values
X_test_e2 = test_df[feature_sets['expert2']].values
X_test_e3 = test_df[feature_sets['expert3']].values

In [17]:
def metrics_dict(y_true, y_pred):
    return {
        'acc': accuracy_score(y_true, y_pred),
        'precision_macro': precision_score(y_true, y_pred, average='macro', zero_division=0),
        'recall_macro': recall_score(y_true, y_pred, average='macro', zero_division=0),
        'f1_macro': f1_score(y_true, y_pred, average='macro', zero_division=0),
        'f1_weighted': f1_score(y_true, y_pred, average='weighted', zero_division=0),
    }


def align_gate_probs(raw_probs, gate_classes, n_experts=3):
    out = np.zeros((raw_probs.shape[0], n_experts), dtype=np.float64)
    for i, cls in enumerate(gate_classes):
        out[:, int(cls)] = raw_probs[:, i]
    return out


def build_oof_meta_features(X, y, base_models, n_classes, n_splits=5, seed=42):
    skf = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=seed)
    n_models = len(base_models)
    oof = np.zeros((X.shape[0], n_models * n_classes), dtype=np.float64)

    for fold, (tr_idx, va_idx) in enumerate(skf.split(X, y), start=1):
        X_tr, X_va = X[tr_idx], X[va_idx]
        y_tr = y[tr_idx]
        for m_idx, (_, model) in enumerate(base_models.items()):
            mdl = clone(model)
            mdl.fit(X_tr, y_tr)
            proba = mdl.predict_proba(X_va)
            start = m_idx * n_classes
            oof[va_idx, start:start + n_classes] = proba
        print(f'Stacking OOF fold {fold}/{n_splits} done')

    return oof


def train_stacking(X_train, y_train, X_val, y_val, n_classes, seed=42):
    base_models = {
        'rf': RandomForestClassifier(
            n_estimators=500,
            random_state=seed,
            n_jobs=-1,
            class_weight='balanced_subsample'
        ),
        'xgb': XGBClassifier(
            n_estimators=400,
            max_depth=6,
            learning_rate=0.05,
            subsample=0.9,
            colsample_bytree=0.9,
            objective='multi:softprob',
            num_class=n_classes,
            tree_method='hist',
            eval_metric='mlogloss',
            random_state=seed,
            n_jobs=-1
        ),
        'et': ExtraTreesClassifier(
            n_estimators=600,
            random_state=seed,
            n_jobs=-1,
            class_weight='balanced'
        ),
        'lr': LogisticRegression(
            max_iter=3000,
            solver='lbfgs',
            random_state=seed
        )
    }

    oof_meta = build_oof_meta_features(
        X_train, y_train, base_models, n_classes, n_splits=N_SPLITS, seed=seed
    )

    fitted_base_models = {}
    val_meta_blocks = []
    for name, model in base_models.items():
        mdl = clone(model)
        mdl.fit(X_train, y_train)
        fitted_base_models[name] = mdl
        val_meta_blocks.append(mdl.predict_proba(X_val))
        print(f'Stacking base model fitted: {name}')

    val_meta = np.hstack(val_meta_blocks)

    meta_candidates = {
        'lr': LogisticRegression(
            max_iter=3000,
            solver='lbfgs',
            random_state=seed
        ),
        'xgb': XGBClassifier(
            n_estimators=300,
            max_depth=4,
            learning_rate=0.05,
            subsample=0.9,
            colsample_bytree=0.9,
            objective='multi:softprob',
            num_class=n_classes,
            tree_method='hist',
            eval_metric='mlogloss',
            random_state=seed,
            n_jobs=-1
        )
    }

    meta_scores = {}
    fitted_metas = {}
    for name, meta in meta_candidates.items():
        m = clone(meta)
        m.fit(oof_meta, y_train)
        val_pred = m.predict(val_meta)
        score = f1_score(y_val, val_pred, average='macro', zero_division=0)
        meta_scores[name] = score
        fitted_metas[name] = m
        print(f'Stacking meta candidate {name} val_macro_f1={score:.6f}')

    best_meta_name = max(meta_scores, key=meta_scores.get)
    best_meta = fitted_metas[best_meta_name]

    def predict_with_stacking(X):
        blocks = [fitted_base_models[name].predict_proba(X) for name in base_models.keys()]
        meta_X = np.hstack(blocks)
        pred = best_meta.predict(meta_X)
        probs = best_meta.predict_proba(meta_X)
        return pred, probs

    val_pred, val_probs = predict_with_stacking(X_val)
    val_metrics = metrics_dict(y_val, val_pred)

    artifacts = {
        'base_models': fitted_base_models,
        'meta_model': best_meta,
        'schema': {
            'base_model_names': list(base_models.keys()),
            'n_classes': n_classes,
            'n_splits': N_SPLITS,
            'meta_candidates_val_macro_f1': meta_scores,
            'selected_meta_model': best_meta_name,
            'selected_features': selected_features
        }
    }

    return artifacts, val_pred, val_probs, val_metrics

In [18]:
def train_moe_diverse(
    X_train_full, y_train, X_val_full, y_val,
    X_train_e1, X_train_e2, X_train_e3,
    X_val_e1, X_val_e2, X_val_e3,
    n_classes, seed=42
):
    expert_defs = {
        'expert1': XGBClassifier(
            n_estimators=450,
            max_depth=5,
            learning_rate=0.05,
            subsample=0.9,
            colsample_bytree=0.9,
            objective='multi:softprob',
            num_class=n_classes,
            tree_method='hist',
            eval_metric='mlogloss',
            random_state=seed,
            n_jobs=-1
        ),
        'expert2': RandomForestClassifier(
            n_estimators=600,
            random_state=seed,
            n_jobs=-1,
            class_weight='balanced_subsample'
        ),
        'expert3': ExtraTreesClassifier(
            n_estimators=900,
            random_state=seed + 7,
            n_jobs=-1,
            class_weight='balanced',
            max_depth=None,
            min_samples_split=2,
            min_samples_leaf=1
        )
    }

    X_train_by_expert = {
        'expert1': X_train_e1,
        'expert2': X_train_e2,
        'expert3': X_train_e3
    }
    X_val_by_expert = {
        'expert1': X_val_e1,
        'expert2': X_val_e2,
        'expert3': X_val_e3
    }

    skf = StratifiedKFold(n_splits=N_SPLITS, shuffle=True, random_state=seed)
    n = X_train_full.shape[0]
    oof_probs = {
        name: np.zeros((n, n_classes), dtype=np.float64)
        for name in expert_defs.keys()
    }

    for fold, (tr_idx, va_idx) in enumerate(skf.split(X_train_full, y_train), start=1):
        y_tr = y_train[tr_idx]
        for name, model in expert_defs.items():
            mdl = clone(model)
            mdl.fit(X_train_by_expert[name][tr_idx], y_tr)
            oof_probs[name][va_idx] = mdl.predict_proba(X_train_by_expert[name][va_idx])
        print(f'MoE OOF fold {fold}/{N_SPLITS} done')

    row_idx = np.arange(n)
    true_class_scores = np.column_stack([
        oof_probs['expert1'][row_idx, y_train],
        oof_probs['expert2'][row_idx, y_train],
        oof_probs['expert3'][row_idx, y_train]
    ])
    gate_targets = true_class_scores.argmax(axis=1)

    gate = MLPClassifier(
        hidden_layer_sizes=(128, 64),
        activation='relu',
        solver='adam',
        alpha=1e-4,
        batch_size=256,
        learning_rate_init=1e-3,
        max_iter=300,
        early_stopping=True,
        random_state=seed
    )
    gate.fit(X_train_full, gate_targets)

    fitted_experts = {}
    for name, model in expert_defs.items():
        mdl = clone(model)
        mdl.fit(X_train_by_expert[name], y_train)
        fitted_experts[name] = mdl
        print(f'MoE expert fitted: {name}')

    def predict_with_moe(X_full, X1, X2, X3):
        raw_gate = gate.predict_proba(X_full)
        gate_probs = align_gate_probs(raw_gate, gate.classes_, n_experts=3)

        p1 = fitted_experts['expert1'].predict_proba(X1)
        p2 = fitted_experts['expert2'].predict_proba(X2)
        p3 = fitted_experts['expert3'].predict_proba(X3)

        final_probs = gate_probs[:, [0]] * p1 + gate_probs[:, [1]] * p2 + gate_probs[:, [2]] * p3
        pred = final_probs.argmax(axis=1)
        return pred, final_probs, gate_probs

    val_pred, val_probs, val_gate_probs = predict_with_moe(
        X_val_full, X_val_e1, X_val_e2, X_val_e3
    )
    val_metrics = metrics_dict(y_val, val_pred)

    artifacts = {
        'experts': fitted_experts,
        'gate': gate,
        'schema': {
            'experts': ['expert1', 'expert2', 'expert3'],
            'feature_sets': feature_sets,
            'n_classes': n_classes,
            'n_splits': N_SPLITS,
            'gate_hidden_layers': [128, 64]
        }
    }

    return artifacts, val_pred, val_probs, val_gate_probs, val_metrics

In [19]:
stacking_artifacts, y_val_pred_stack, val_probs_stack, stack_val_metrics = train_stacking(
    X_train=X_train,
    y_train=y_train,
    X_val=X_val,
    y_val=y_val,
    n_classes=num_classes,
    seed=SEED
)

moe_artifacts, y_val_pred_moe, val_probs_moe, gate_probs_val_moe, moe_val_metrics = train_moe_diverse(
    X_train_full=X_train,
    y_train=y_train,
    X_val_full=X_val,
    y_val=y_val,
    X_train_e1=X_train_e1,
    X_train_e2=X_train_e2,
    X_train_e3=X_train_e3,
    X_val_e1=X_val_e1,
    X_val_e2=X_val_e2,
    X_val_e3=X_val_e3,
    n_classes=num_classes,
    seed=SEED
)

print('Stacking val metrics:', stack_val_metrics)
print('MoE val metrics     :', moe_val_metrics)
print('\nStacking report')
print(classification_report(y_val, y_val_pred_stack, target_names=class_names, zero_division=0))
print('\nMoE report')
print(classification_report(y_val, y_val_pred_moe, target_names=class_names, zero_division=0))

Stacking OOF fold 1/5 done
Stacking OOF fold 2/5 done
Stacking OOF fold 3/5 done
Stacking OOF fold 4/5 done
Stacking OOF fold 5/5 done
Stacking base model fitted: rf
Stacking base model fitted: xgb
Stacking base model fitted: et
Stacking base model fitted: lr
Stacking meta candidate lr val_macro_f1=0.543323
Stacking meta candidate xgb val_macro_f1=0.453836
MoE OOF fold 1/5 done
MoE OOF fold 2/5 done
MoE OOF fold 3/5 done
MoE OOF fold 4/5 done
MoE OOF fold 5/5 done
MoE expert fitted: expert1
MoE expert fitted: expert2
MoE expert fitted: expert3
Stacking val metrics: {'acc': 0.7688271692948188, 'precision_macro': 0.6305840786775012, 'recall_macro': 0.5068214527819627, 'f1_macro': 0.5433230235542819, 'f1_weighted': 0.7934861083487994}
MoE val metrics     : {'acc': 0.7750149704867547, 'precision_macro': 0.614513555346069, 'recall_macro': 0.5882734429895041, 'f1_macro': 0.5885719632346413, 'f1_weighted': 0.8053697943716489}

Stacking report
                precision    recall  f1-score   su

In [20]:
winner_model = 'stacking' if stack_val_metrics['f1_macro'] >= moe_val_metrics['f1_macro'] else 'moe'

joblib.dump(stacking_artifacts['base_models'], os.path.join(models_dir, 'stacking_base_models.pkl'))
joblib.dump(stacking_artifacts['meta_model'], os.path.join(models_dir, 'stacking_meta_model.pkl'))
joblib.dump(stacking_artifacts['schema'], os.path.join(models_dir, 'stacking_schema.pkl'))

joblib.dump(moe_artifacts['experts']['expert1'], os.path.join(models_dir, 'moe_expert1_xgb.pkl'))
joblib.dump(moe_artifacts['experts']['expert2'], os.path.join(models_dir, 'moe_expert2_rf.pkl'))
joblib.dump(moe_artifacts['experts']['expert3'], os.path.join(models_dir, 'moe_expert3_et.pkl'))
joblib.dump(moe_artifacts['gate'], os.path.join(models_dir, 'moe_gate_mlp.pkl'))
joblib.dump(moe_artifacts['schema'], os.path.join(models_dir, 'moe_schema.pkl'))

best_model_meta = {
    'winner_model': winner_model,
    'winner_metric': 'f1_macro',
    'stacking_val_metrics': stack_val_metrics,
    'moe_val_metrics': moe_val_metrics,
    'artifact_paths': {
        'stacking_base_models': 'stacking_base_models.pkl',
        'stacking_meta_model': 'stacking_meta_model.pkl',
        'stacking_schema': 'stacking_schema.pkl',
        'moe_expert1': 'moe_expert1_xgb.pkl',
        'moe_expert2': 'moe_expert2_rf.pkl',
        'moe_expert3': 'moe_expert3_et.pkl',
        'moe_gate': 'moe_gate_mlp.pkl',
        'moe_schema': 'moe_schema.pkl'
    }
}
joblib.dump(best_model_meta, os.path.join(models_dir, 'best_model_meta.pkl'))

print('Winner model:', winner_model)

Winner model: moe


In [21]:
def predict_stacking(artifacts, X):
    base_models = artifacts['base_models']
    schema = artifacts['schema']
    meta = artifacts['meta_model']
    blocks = [base_models[name].predict_proba(X) for name in schema['base_model_names']]
    meta_X = np.hstack(blocks)
    y_pred = meta.predict(meta_X)
    probs = meta.predict_proba(meta_X)
    return y_pred, probs


def predict_moe(artifacts, X_full, X1, X2, X3):
    experts = artifacts['experts']
    gate = artifacts['gate']
    raw_gate = gate.predict_proba(X_full)
    gate_probs = align_gate_probs(raw_gate, gate.classes_, n_experts=3)

    p1 = experts['expert1'].predict_proba(X1)
    p2 = experts['expert2'].predict_proba(X2)
    p3 = experts['expert3'].predict_proba(X3)

    probs = gate_probs[:, [0]] * p1 + gate_probs[:, [1]] * p2 + gate_probs[:, [2]] * p3
    y_pred = probs.argmax(axis=1)
    return y_pred, probs


if winner_model == 'stacking':
    y_test_pred, test_probs = predict_stacking(stacking_artifacts, X_test)
else:
    y_test_pred, test_probs = predict_moe(
        moe_artifacts, X_test, X_test_e1, X_test_e2, X_test_e3
    )

history = {
    'stacking_val': stack_val_metrics,
    'moe_val': moe_val_metrics,
    'winner_model': winner_model,
    'winner_val_metrics': stack_val_metrics if winner_model == 'stacking' else moe_val_metrics
}

np.save(os.path.join(models_dir, 'preds.npy'), y_test_pred)
np.save(os.path.join(models_dir, 'true.npy'), y_test)

with open(os.path.join(models_dir, 'history.pkl'), 'wb') as f:
    pickle.dump(history, f)

print('Saved new ensemble artifacts, best_model_meta.pkl, preds.npy, true.npy, history.pkl')

Saved new ensemble artifacts, best_model_meta.pkl, preds.npy, true.npy, history.pkl
